# EXP-2026-001 on Colab: crop + 512 px representation gate

Variant `crop192_512_class_rgb`, added to the preregistered order after the first three fallbacks failed and before it ran:
- **Crop:** a 192 × 192 mm in-plane window centred on each patient's own pre-op brain, so it uses pre-op data only. The size fits every training brain.
- **Resolution:** resampled to 512 px, so 0.375 mm per pixel and a 3 mm VAE cell.
- **Mask encoding:** one class per color channel, same rules as before.
- **Scoring:** back in the 1 mm grid, with every threshold unchanged.

Selection rule: the first variant passing on every path in the order 512_scalar, 256_class_rgb, 512_class_rgb, crop192_512_class_rgb is selected.

Nothing needs uploading for this run; the test step uses whatever snapshot is already on Drive. Use the GPU runtime. **Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "exp-2026-001-implementation"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
WORK = "/content/MedSegFactory"
RUN_TESTS = True  # optional environment check; never blocks the gate

## 1. Mount Drive, get the latest code from the fork, link the data

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
assert (PRIVATE / "split/split.json").exists(), "The synthesis folder is missing on Drive"
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 2. Dependencies and (optional) tests

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "nibabel>=5.3,<6", "SimpleITK==2.5.6", "pydicom>=3,<4", "scikit-image>=0.25"], check=True)

if RUN_TESTS:
    import tarfile

    snapshots = PRIVATE / "test_snapshots"
    pointer = snapshots / "LATEST.txt"
    name = pointer.read_text(encoding="utf-8").strip() if pointer.exists() else ""
    if not (snapshots / name).is_file():
        archives = sorted(snapshots.glob("tests-*.tar.gz"), key=os.path.getmtime)
        name = archives[-1].name if archives else ""
    if name:
        with tarfile.open(snapshots / name) as archive:
            try:
                archive.extractall(WORK, filter="data")
            except TypeError:
                archive.extractall(WORK)
        result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
        print("tests snapshot:", name, "| this snapshot may predate the newest tests")
        print(result.stdout[-1200:])
    else:
        print("No tests snapshot on Drive; skipping (tests are run on the laptop before every push)")

## 3. Run the crop gate (GPU)

In [ ]:
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
report_path = PRIVATE / "representation_gate_crop192_512_class_rgb" / "report.json"
if report_path.exists():
    print("crop192_512_class_rgb: report already exists, not rerun (write-once)")
else:
    command = [sys.executable, "scripts/representation_gate.py", "--allow-download", "--device", "cuda",
               "--codec", "class_separated_rgb", "--crop-mm", "192", "--out-px", "512", "--batch", "8",
               "--exclude", *policy["excluded_training"]]
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout[-1500:])
    if result.returncode:
        print(result.stderr[-3000:])

## 4. Compare all variants and apply the selection rule

In [ ]:
import pandas as pd

ORDER = ["512_scalar", "256_class_rgb", "512_class_rgb", "crop192_512_class_rgb"]
folders = {"256_scalar (reference, v1)": "representation_gate_v1"}
folders.update({name: f"representation_gate_{name}" for name in ORDER})
rows, passed = [], {}
for label, folder in folders.items():
    path = PRIVATE / folder / "report.json"
    if not path.exists():
        print("missing:", label)
        continue
    report = json.loads(path.read_text(encoding="utf-8"))
    passed[label] = report["pass"]
    s = report["paths"]["mean"]
    worst = {key: min(p[key] for p in report["paths"].values()) for key in ("macro_id3_dice", "macro_id3_recall")}
    rows.append({
        "variant": label,
        "id3_dice (mean / worst path)": f"{s['macro_id3_dice']:.3f} / {worst['macro_id3_dice']:.3f}",
        "id3_recall (mean / worst path)": f"{s['macro_id3_recall']:.3f} / {worst['macro_id3_recall']:.3f}",
        "lesions_kept": f"{s['pooled_matched_lesions']}/{s['pooled_reference_lesions']}",
        "fake_id3_ml": round(max(p["mean_spurious_id3_ml_negative"] for p in report["paths"].values()), 4),
        "fake_id1_ml": round(max(p["mean_unsupported_id1_ml"] for p in report["paths"].values()), 4),
        "pass_all_paths": report["pass"],
    })
crop_report = PRIVATE / "representation_gate_crop192_512_class_rgb" / "report.json"
if crop_report.exists():
    qc = json.loads(crop_report.read_text(encoding="utf-8"))["crop_qc"]
    flagged = sum(1 for q in qc.values() if q["excluded_foreground_voxels"] or q["excluded_reference_label_voxels"])
    print(f"crop coverage: {len(qc) - flagged}/{len(qc)} training patients fully covered")
selected = next((name for name in ORDER if passed.get(name)), None)
print("Selected variant:", selected or "none: representation failure")
pd.DataFrame(rows)

## Next

Send Claude the "Selected variant" line, the crop coverage line and the table (aggregate numbers only).